In [73]:
import sys;sys.path.append('../')
import scipy.sparse as sp
import numpy as np
from matplotlib import pyplot as plt

from modules import Hamiltonian as ham
from modules import PauliStrings
from modules import Evolution as evol
from pathlib import Path

import importlib
importlib.reload(evol)

outputs_dir = Path('../outputs')
plots_dir = Path('../plots')

In [2]:
#Hamiltonian Params Traverse Field Ising model
n_qubits=8
J=0.5
h=1
T=2

#trotter params
H,H_trot=ham.TFIM(J,h,n_qubits,T=T)
print('Computing Eigs')
EH,VH = np.linalg.eigh(H.todense())
psigs=VH[:,0]
E_gs=EH[0]

#evolution parameters
D=4
dt=0.05
NQ=20 #maximum number of steps in QITE

#initial state
psi0np=np.zeros((2**n_qubits,1),dtype=complex)
psi0np[0]=1
psi_0=sp.csc_matrix(psi0np/np.linalg.norm(psi0np))



Computing Eigs


In [64]:
#QITE without Lasso
print("Computing QITE")
EQ_vanilla,psi_QITE_vanilla,a_vanilla = evol.QITE(n_qubits,H,H_trot,D,psi_0,NQ,dt)

Computing QITE


In [80]:
#QITE with vanilla Lasso
print("Computing QITE with vanilla Lasso")
EQ_lasso,psi_QITE_lasso,a_lasso = evol.QITE(n_qubits,H,H_trot,D,psi_0,NQ,dt,method='vanilla_lasso',lasso_lambda=0.05)

Computing QITE with vanilla Lasso


## Gate counts

In [74]:
from modules import Transpile
importlib.reload(Transpile)
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.quantum_info import Statevector,state_fidelity

In [6]:
if H_trot.BC == 'OBC':
    if D == n_qubits:
        num_paulis,PD,fail = PauliStrings.OBC_DN(H_trot,n_qubits,PDstr=True)
    elif np.isreal(H.data).all() and np.isreal(psi_0.data).all():
        print("Using Real Pauli Strings")
        num_paulis,PD,fail = PauliStrings.real_OBC(H_trot,D,n_qubits,PDstr=True)
    else:
        print("Using General Pauli Strings")
        num_paulis,PD,fail = PauliStrings.general_OBC(H_trot,D,n_qubits,PDstr=True)
elif np.isreal(H.data).all() and np.isreal(psi_0.data).all():
    print("Using Real Pauli Strings")
    num_paulis,PD,fail = PauliStrings.real(H_trot,D,n_qubits,PDstr=True)
else:
    print("Using General Pauli Strings")
    num_paulis,PD,fail = PauliStrings.general(H_trot,D,n_qubits,PDstr=True)

Using Real Pauli Strings


In [7]:
backend1 = GenericBackendV2(num_qubits=n_qubits,basis_gates=['u3','cx'])

In [75]:
tr_qc_vanilla,true_vanilla,error_vanilla = Transpile.transpile_circuit_qite(n_qubits,D,T,PD,a_vanilla,dt,NQ,backend1,1,PEvolution=True)

In [81]:
tr_qc_lasso,true_lasso,error_lasso = Transpile.transpile_circuit_qite(n_qubits,D,T,PD,a_lasso,dt,NQ,backend1,1,PEvolution=True)

In [82]:
tr_qc_vanilla.count_ops(), tr_qc_lasso.count_ops()

(OrderedDict([('cx', 45288), ('u3', 40044)]),
 OrderedDict([('u3', 2517), ('cx', 2292)]))

In [83]:
psi_final_QITE_tr = Statevector(psi0np).evolve(tr_qc_vanilla)
psi_QITE_vanilla_final = psi_QITE_vanilla[:,-1].todense()

psi_final_lasso_tr = Statevector(psi0np).evolve(tr_qc_lasso)
psi_QITE_lasso_final = psi_QITE_lasso[:,-1].todense()

In [84]:
state_fidelity(psi_final_QITE_tr,psi_QITE_vanilla_final), state_fidelity(psi_final_lasso_tr,psi_QITE_lasso_final)

(0.9998321832302226, 0.9999239639659138)

In [85]:
state_fidelity(psi_QITE_vanilla_final, psigs), state_fidelity(psi_QITE_lasso_final, psigs)

(0.9631064474802163, 0.9688317204980081)